# Inicialização

In [1]:
from pathlib import Path
import os
import logging

## ---------LOGGING-----------
LOGGER = logging

# Define paths based on environment
if Path("/kaggle").exists():
    LOGGER.info("IN KAGGLE")
    os.environ["AMBIENTE"] = "KAGGLE"
    os.environ["TENSORBOARD_NO_TF"] = "1"

    PATH_DATASET = Path("/kaggle/working/STROKE_PREDICTION")
    PATH_CODE = PATH_DATASET / "src"
    PATH_OUTPUT_DIR = PATH_DATASET / "outputs"
    EXPORT_DIR = PATH_OUTPUT_DIR / "export"
elif Path("/content").exists():
    os.environ["AMBIENTE"] = "COLAB"
    # PATH_DATASET = Path("/content/DELETAR")
    # PATH_CODE = PATH_DATASET / "src"
    # PATH_OUTPUT_DIR = PATH_DATASET / "outputs"
    # EXPORT_DIR = PATH_OUTPUT_DIR / "export"
    raise Exception
else:
    os.environ["AMBIENTE"] = "LOCAL"
    PATH_CODE = Path.cwd()
    PATH_DATASET = PATH_CODE
    PATH_OUTPUT_DIR = PATH_DATASET / "outputs"
    # Add to setup cell
    EXPORT_DIR = PATH_OUTPUT_DIR / "export"

assert type(PATH_DATASET) is not str, "PATH_DATASET NÃO DEVE SER STR!"
# Check if installation has been done
INSTALL_MARKER = PATH_DATASET / ".install_complete"

try:
    if not INSTALL_MARKER.exists():
        # Install uv
        pass
        !pip install uv

        # Environment-specific setup
        if os.environ["AMBIENTE"] == "KAGGLE":
            import kaggle_secrets

            user_secrets = kaggle_secrets.UserSecretsClient()
            github_pat = user_secrets.get_secret("GITHUB_PAT")

            os.chdir("/kaggle/working")
            os.system(
                f"git clone -b class-imbalance https://{github_pat}@github.com/lfaoliveira/TRAB_SERIES_TEMP.git"
            )
            os.chdir(PATH_DATASET)

        # Install dependencies
        os.chdir(PATH_DATASET)
        os.system("uv pip install --requirements pyproject.toml --system")

        if os.environ["AMBIENTE"] == "KAGGLE":
            os.system(
                "uv pip install --upgrade --force-reinstall --no-cache-dir scipy numpy matplotlib protobuf tensorboard"
            )

        # Mark installation as complete
        INSTALL_MARKER.touch()
        LOGGER.info("Installation completed")
    else:
        LOGGER.info("Installation already completed, skipping...")

    os.chdir(PATH_CODE)
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    LOGGER.info(f"Current working directory: {os.getcwd()}")
except Exception as e:
    LOGGER.info("FALHA AO INICIAR NOTEBOOK")
    LOGGER.info(e)

## Carregamento

In [2]:
import os
from pathlib import Path

import matplotlib.pyplot as plt

from src.data.dataset import StrokeDataset
from src.data.datamodule import StrokeDataModule

# ============================================================
# CARREGAMENTO E EXPLORAÇÃO DOS DADOS
# ============================================================

# Instanciar dataset (carrega e preprocessa automaticamente)
print("Carregando StrokeDataset...")
PATH_DATA = Path("data")
PATH_DATA.mkdir(exist_ok=True)
dataset = StrokeDataset(PATH_DATA / "m4")

print(dataset.build_train_dataset().data)


print("\n✓ Dataset carregado com sucesso!")


# ============================================================
# EXPLORAÇÃO E VISUALIZAÇÃO
# ============================================================


def explorar_dataset(dataset: StrokeDataset, name: str = "M4") -> None:
    """
    Exibe informações e visualizações do Dataset.

    Args:
        dataset: StrokeDataset carregado
        name: Nome do dataset para exibição
    """
    print("\n" + "=" * 60)
    print(f"INFORMAÇÕES DO {name}")

    print("=" * 60)
    print(f"Frequência: {dataset.frequency}")
    print(f"Input Width: {dataset.frequency}")
    print(f"Horizon (Output Width): {dataset.frequency}")
    print(f"Número de amostras de treino: {len(dataset.train_dataset.index)}")
    print(f"Dimensionalidade dos dados: {dataset.train_dataset.data}")

    # Visualizar algumas amostras
    data_sample = dataset.data[:5].numpy()
    labels_sample = dataset.labels_train[:, :]

    print("\nAmostra de dados (primeiras 5 séries, primeiros 10 timestamps):")
    print(data_sample[:, :10])

    print("\nAmostra de labels (primeiras 5 séries):")
    print(labels_sample)

    # Estatísticas dos dados
    print("\nESTATÍSTICAS DOS DADOS")
    print(f"  - Média: {data_sample.mean():.4f}")
    print(f"  - Std: {data_sample.std():.4f}")
    print(f"  - Min: {data_sample.min():.4f}")
    print(f"  - Max: {data_sample.max():.4f}")

    # Visualizações
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Distribuição dos dados
    axes[0].hist(
        data_sample.ravel(), bins=50, color="#0B4F6C", alpha=0.7, edgecolor="black"
    )
    axes[0].set_title("Distribuição dos dados (normalizado)", fontweight="bold")
    axes[0].set_xlabel("Valor")
    axes[0].set_ylabel("Frequência")
    axes[0].grid(True, alpha=0.3)

    # Série temporal de exemplo
    axes[1].plot(data_sample[0], marker="o", linewidth=2, markersize=4, label="Série 1")
    axes[1].plot(data_sample[1], marker="s", linewidth=2, markersize=4, label="Série 2")
    axes[1].set_title("Exemplos de séries temporais", fontweight="bold")
    axes[1].set_xlabel("Timestamp")
    axes[1].set_ylabel("Valor (normalizado)")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


# Executar exploração
explorar_dataset(dataset, name="M4 Dataset")

run_mode='prototype' dataset_frequency='Daily'
Carregando StrokeDataset...
DADOS BAIXADOS E LIDOS!!
DATASET DE TREINO CRIADO!
DATASET DE TESTE CRIADO!
DATASET DE VALIDAÇÂO CRIADO!
{'reals': tensor([[ 0.0000, -1.7319, -0.1548],
        [ 0.0000, -1.7315, -0.1083],
        [ 0.0000, -1.7312, -0.0331],
        ...,
        [ 0.0000,  1.7312, -3.7758],
        [ 0.0000,  1.7315, -3.5340],
        [ 0.0000,  1.7319, -3.5967]]), 'categoricals': tensor([], size=(9919, 0), dtype=torch.int64), 'groups': tensor([[0],
        [0],
        [0],
        ...,
        [0],
        [0],
        [0]]), 'target': [tensor([1043.1000, 1044.4000, 1046.5000,  ...,  942.0000,  948.7500,
         947.0000])], 'weight': None, 'time': tensor([   1,    2,    3,  ..., 9917, 9918, 9919])}

✓ Dataset carregado com sucesso!

INFORMAÇÕES DO M4 Dataset
Frequência: Daily
Input Width: Daily
Horizon (Output Width): Daily
Número de amostras de treino: 9899
Dimensionalidade dos dados: {'reals': tensor([[ 0.0000, -1.7319, -

AttributeError: 'StrokeDataset' object has no attribute 'data'

# Exploratory Data Analysis

# Treinamento dos modelos

In [ ]:
import gc
import logging
import os
from pathlib import Path
from typing import Literal

import mlflow
import torch
from lightning import Trainer, seed_everything
from lightning.pytorch.callbacks import EarlyStopping
from lightning.pytorch.loggers import MLFlowLogger
from mlflow.pytorch import autolog
from src.models.kan import KANSearchSpace, MyKan
from src.models.mlp import MLP, MLPSearchSpace


def get_best_hyperparameters_from_optuna(choice: str, mlf_track_uri: str) -> dict:
    """
    Reads the best run ID from the CSV file saved during Optuna training,
    fetches the run from MLflow, and returns the hyperparameters as a dict.

    Args:
        choice: Model architecture name (e.g., "MLP", "KAN")
        mlf_track_uri: MLflow tracking URI

    Returns:
        Dictionary of hyperparameters from the best Optuna run (with string keys)
    """
    return {}  # Placeholder - implement as needed


def zip_res_simplified(
    export_dir: Path,
    filename: str,
    dest_folder: Path | None = None,
    mlflow_db_path: Path | None = None,
    optuna_db_path: Path | None = None,
    mlruns_path: Path | None = None,
):
    """
    Simplified zip function that copies databases and mlruns to export_dir, then zips everything.

    Args:
        export_dir: Directory containing all files to zip (artifacts, CSVs, etc.)
        filename: Name of the output zip file
        dest_folder: Destination folder for the zip file
        mlflow_db_path: Path to mlflow.db to copy into export_dir
        optuna_db_path: Path to optuna.db to copy into export_dir
        mlruns_path: Path to mlruns folder to copy into export_dir
    """
    import shutil

    # Ensure export_dir exists
    export_dir.mkdir(parents=True, exist_ok=True)

    # Copy databases if they exist
    if mlflow_db_path and mlflow_db_path.exists():
        shutil.copy(mlflow_db_path, export_dir / mlflow_db_path.name)
        print(f"Copied {mlflow_db_path.name} to export directory")

    if optuna_db_path and optuna_db_path.exists():
        shutil.copy(optuna_db_path, export_dir / optuna_db_path.name)
        print(f"Copied {optuna_db_path.name} to export directory")

    # Copy mlruns folder if it exists
    if mlruns_path and mlruns_path.exists():
        export_mlruns = export_dir / mlruns_path.name
        if export_mlruns.exists():
            shutil.rmtree(export_mlruns)
        shutil.copytree(mlruns_path, export_mlruns)
        print(f"Copied {mlruns_path.name} folder to export directory")

    # Determine destination folder
    if dest_folder is None:
        dest_folder = Path.cwd()
    else:
        dest_folder.mkdir(parents=True, exist_ok=True)

    # Create zip file
    zip_path = dest_folder / filename.replace(".zip", "")
    shutil.make_archive(str(zip_path), "zip", export_dir)
    print(f"PATH ZIPFILE: {zip_path.with_suffix('.zip').resolve()}")


def supress_warnings():
    # Suppress specific MLflow warnings
    logging.getLogger("mlflow.utils.requirements_utils").setLevel(logging.ERROR)
    logging.getLogger("mlflow").setLevel(logging.ERROR)

    # Suppress PyTorch Lightning info messages
    logging.getLogger("pytorch_lightning.utilities.rank_zero").setLevel(logging.ERROR)


def model_choice(
    CHOICE,
    INPUT_DIMS,
    N_CLASSES,
    recall_factor: list[float] | None = None,
    optuna_hyperparams: dict[str, float | int] = {},
):
    """
    Seleciona e configura o modelo baseado na escolha.

    Args:
        CHOICE: Arquitetura do modelo ("MLP" ou "KAN")
        INPUT_DIMS: Dimensionalidade da entrada
        N_CLASSES: Número de classes
        recall_factor: Fator de recall para balanceamento (opcional)
        optuna_hyperparams: Hiperparâmetros do Optuna

    Returns:
        Tuple de (model, suggested_hparams, keys)
    """
    if recall_factor is None:
        recall_factor = [1.0, 1.0]

    if CHOICE == "MLP":
        search_space = MLPSearchSpace()

        keys = search_space.Keys
        hyperparams = {
            keys.BATCH_SIZE: 32,
            keys.HIDDEN_DIMS: 512,
            keys.LR: 2e-7,
            keys.WEIGHT_DECAY: 1e-5,
            keys.BETA0: 0.99,
            keys.BETA1: 0.999,
            keys.N_LAYERS: 80,
        }

        # Map optuna hyperparams (string keys) to Keys enum
        if optuna_hyperparams:
            mapped_hyperparams = {}
            for key_enum, default_value in hyperparams.items():
                # Get the string representation of the enum key
                key_str = key_enum.value
                if key_str in optuna_hyperparams:
                    mapped_hyperparams[key_enum] = optuna_hyperparams[key_str]
                else:
                    mapped_hyperparams[key_enum] = default_value
            hyperparams = mapped_hyperparams

        print(f"MLP HYPERPARAMS: {hyperparams}")
        suggested_hparams = search_space.suggest(hyperparams)
        model = MLP(
            INPUT_DIMS,
            N_CLASSES,
            recall_factor=recall_factor,
            hyperparameters=suggested_hparams,
        )
    elif CHOICE == "KAN":
        search_space = KANSearchSpace()
        keys = search_space.Keys
        hyperparams = {
            keys.BATCH_SIZE: 32,
            keys.HIDDEN_DIMS: 100,
            keys.LR: 2e-7,
            keys.WEIGHT_DECAY: 1e-5,
            keys.BETA0: 0.99,
            keys.BETA1: 0.999,
            keys.GRID: 184,
            keys.SPLINE_POL_ORDER: 4,
        }

        # Map optuna hyperparams (string keys) to Keys enum
        if optuna_hyperparams:
            mapped_hyperparams = {}
            for key_enum, default_value in hyperparams.items():
                # Get the string representation of the enum key
                key_str = key_enum.value
                if key_str in optuna_hyperparams:
                    mapped_hyperparams[key_enum] = optuna_hyperparams[key_str]
                else:
                    mapped_hyperparams[key_enum] = default_value
            hyperparams = mapped_hyperparams

        print(f"KAN HYPERPARAMS: {hyperparams}")
        suggested_hparams = search_space.suggest(hyperparams)
        model = MyKan(
            INPUT_DIMS,
            N_CLASSES,
            recall_factor=recall_factor,
            hyperparameters=suggested_hparams,
        )
    else:
        raise ValueError("ESCOLHA DE MODELO ERRADA!")
    return model, suggested_hparams, keys


## ======================== FUNÇÃO PRINCIPAL ========================


def main(CHOICE: str):
    """
    Função principal de treinamento.

    Args:
        CHOICE: Arquitetura do modelo ("MLP" ou "KAN")
    """
    ###------SEEDS---------###
    RAND_SEED = 42
    seed_everything(RAND_SEED)
    supress_warnings()

    AMBIENTE = os.environ["AMBIENTE"]
    GPU = True if AMBIENTE in ["KAGGLE", "COLAB"] else False

    ## ---------- VARIÁVEIS DE TREINAMENTO -----------
    cpus = os.cpu_count()
    WORKERS = cpus if cpus is not None else 1
    NUM_DEVICES = 1 if GPU else 1
    NUM_NODES = 1
    BATCH_SIZE = 64
    EPOCHS = 1
    PATIENCE = int(EPOCHS * 0.6)
    ARTIFACT_PATH = EXPORT_DIR / "artifacts"
    os.makedirs(ARTIFACT_PATH, exist_ok=True)

    #### -------- VARIÁVEIS DE LOGGING ------------
    EXP_NAME = "PROD_TRAINING"
    RUN_NAME: str | None = f"PROD_{CHOICE}"
    MLF_TRACK_URI = f"sqlite:///{PATH_CODE}/mlflow.db"

    mlflow.set_tracking_uri(MLF_TRACK_URI)
    mlflow.set_experiment(EXP_NAME)
    autolog(log_models=True, checkpoint=True, exclusive=False)

    hyperparams = get_best_hyperparameters_from_optuna(CHOICE, MLF_TRACK_URI)

    ## ---------- VARIÁVEIS DO MODELO -----------
    N_CLASSES = 2

    # Preparar datamodule
    datamodule = StrokeDataModule(BATCH_SIZE, WORKERS)
    datamodule.prepare_data()
    datamodule.setup("fit")

    INPUT_DIMS = datamodule.input_dims or -1
    assert INPUT_DIMS > 0 and INPUT_DIMS is not None

    # Criar modelo
    model, _, keys = model_choice(
        CHOICE,
        INPUT_DIMS,
        N_CLASSES,
        recall_factor=None,  # Sem balanceamento específico por enquanto
        optuna_hyperparams=hyperparams,
    )

    _ = model(model.example_input_array)

    # Loop principal de treinamento
    with mlflow.start_run(run_name=RUN_NAME) as run:
        active_run_id = run.info.run_id

        mlflow_logger = MLFlowLogger(
            experiment_name=EXP_NAME,
            tracking_uri=MLF_TRACK_URI,
            log_model=True,
            run_id=active_run_id,
        )

        early_stopping = EarlyStopping(
            monitor="val_loss", patience=PATIENCE, mode="min"
        )

        trainer = Trainer(
            max_epochs=EPOCHS,
            devices=NUM_DEVICES,
            accelerator="gpu" if GPU else "cpu",
            num_nodes=NUM_NODES,
            logger=mlflow_logger,
            enable_checkpointing=False,
            log_every_n_steps=1,
            callbacks=[early_stopping],
        )
        trainer.fit(model, datamodule=datamodule)
        mlflow.log_params(dict(model.hparams))

        # Test the model
        test_loader = datamodule.test_dataloader()
        trainer.test(model, dataloaders=test_loader)

        torch.cuda.empty_cache()

    return


if __name__ == "__main__":
    try:
        ARQ_TYPE = Literal["MLP", "KAN", "SVM", "XGBOOST"]  ## MODEL ARCHITECTURE
        models: list[ARQ_TYPE] = ["MLP", "KAN"]
        for choice in models:
            # trains model based on architecture
            if choice == "MLP":
                continue
            main(choice)

        NAME_RESZIP = "resultado_kaggle_stroke_normal"
        MLRUNS_FOLDER = PATH_CODE / "mlruns"
        MLF_TRACK_URI = f"sqlite:///{PATH_CODE}/mlflow.db"
        OPTUNA_DB_PATH = PATH_CODE / "optuna.db"
        ZIP_ROOT = (
            PATH_DATASET / ".." if os.environ["AMBIENTE"] == "KAGGLE" else PATH_DATASET
        )

        zip_res_simplified(
            export_dir=EXPORT_DIR,
            filename=NAME_RESZIP,
            dest_folder=ZIP_ROOT,
            mlflow_db_path=Path(MLF_TRACK_URI.replace("sqlite:///", "")),
            optuna_db_path=OPTUNA_DB_PATH,
            mlruns_path=MLRUNS_FOLDER,
        )
        print("\n", "=" * 60)
        print(f"RESULTADOS ZIPADOS {Path(ZIP_ROOT, NAME_RESZIP).resolve()}")
        print("=" * 60, "\n")

    except Exception as e:
        raise e
    gc.collect()

    if os.environ["AMBIENTE"] == "LOCAL":
        from view.dashboard import see_model

        see_model(PATH_DATASET / "mlflow.db", PATH_DATASET / ".." / "mlruns")

## Teste dos modelos

## Otimização de hiperparâmetros

## Teste da otimização

## OUTPUT: MLFlow's Dashboard (Only works outside of Kaggle)
### Download the training results from Kaggle and paste them into a cloned folder of the repository